<a href="https://colab.research.google.com/github/carmenbonal/2526_Computacional/blob/main/codigos/ising/ISING.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## ISING


**Resumen**

El modelo de Ising estudia las transiciones de fase magnéticas mediante una red de espines que interactúan con sus vecinos. Utilizando el algoritmo de Metropolis, se generan configuraciones de equilibrio para calcular magnitudes como la magnetización y la energía, analizando el comportamiento del sistema frente a cambios en la temperatura.


**1. Fundamento teórico**

**2. Metodología y herramientas**

La implementación de este simulador se ha realizado mediante una colaboración con el modelo de lenguaje Gemini (IA), que ha actuado como una herramienta de apoyo para el desarrollo de un motor de simulación de Física Estadística en Python. A través de este proceso, la IA facilitó la integración del algoritmo de Metropolis para garantizar que el sistema evolucione hacia la distribución de equilibrio de Boltzmann y el manejo preciso de las interacciones entre espines en una red bidimensional. Además de optimizar el cálculo de la diferencia de energía ($\Delta E$) y el cumplimiento de las condiciones de contorno periódicas, Gemini asistió en la estructuración de un fichero de datos compatible con el script 'animacion_ising.py', permitiendo transformar las ecuaciones de interacción magnética en una animación dinámica mediante la librería matplotlib.

 **2.1 Prompt de Diseño**

Para asegurar la obtención de un código funcional y riguroso que resuelva el problema de Ising, se establecieron las siguientes especificaciones:

*Objetivo:* Implementación del algoritmo de Metropolis:Inicialización del sistema:

 Se elige un punto al azar $(n, m)$ de la red de tamaño $N \times N$ en cada intento de cambio. Evaluamos el cambio de energía: $$\Delta E = 2s(n,m) \sum_{vecinos} s_{vecino}$$

 *Criterio de aceptación:* Generación de un número aleatorio uniforme $\xi \in [0,1]$ para decidir el cambio de signo del espín según la probabilidad $p = \min(1, e^{-\beta \Delta E})$, asegurando la convergencia.


 *Entorno y librerías:* Uso de NumPy para la gestión eficiente del retículo de espines y el cálculo matricial. Generación de un fichero de salida estructurado por bloques para su visualización con el script de animación proporcionado.

**3. Resultados y discusión**

**3.1. Posiblemes régimen a distintas temperaturas**
Usando el código explicado para el mismo tamaño de red (N=32) se estudiaron los tres posibles régimen del sistema:
1. Baja Temperatura ($T = 1.5$) (Fase Ferromagnética)
2. Temperatura crítica ($T = 2.269$) (Temperatura Crítica - $T_c$)
3. Alta Temperatura ($T = 4.0$) (Fase Paramagnética)

In [2]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

# --- 1. Funciones de la Simulación (Modelo de Ising) ---

def inicializar_sistema(n, ordenada=False):
    if ordenada:
        return np.ones((n, n), dtype=int)
    else:
        return np.random.choice([-1, 1], size=(n, n))

def calcular_delta_e(grid, i, j, n):
    s = grid[i, j]
    # Condiciones de contorno periódicas
    vecinos = (grid[(i + 1) % n, j] + grid[(i - 1) % n, j] +
               grid[i, (j + 1) % n] + grid[i, (j - 1) % n])
    return 2 * s * vecinos

def paso_metropolis(grid, n, temp):
    for _ in range(n**2):
        i, j = np.random.randint(0, n, size=2)
        de = calcular_delta_e(grid, i, j, n)

        if de <= 0:
            grid[i, j] *= -1
        elif np.random.rand() < np.exp(-de / temp):
            grid[i, j] *= -1
    return grid

def ejecutar_simulacion(n, temp, pasos_mc, frec_guardado):
    """Ejecuta la simulación y devuelve una lista con los fotogramas."""
    grid = inicializar_sistema(n, ordenada=False)
    fotogramas = []

    for paso in range(pasos_mc):
        grid = paso_metropolis(grid, n, temp)
        if paso % frec_guardado == 0:
            # Es vital guardar una COPIA del grid (.copy())
            fotogramas.append(grid.copy())
    return fotogramas

# --- 2. Parámetros Globales ---

N = 32               # Tamaño del retículo
PASOS_MC = 500       # Pasos totales
FREC_GUARDADO = 5    # Guardar cada X pasos
TEMPERATURAS = [1.5, 2.269, 4.0]  # Baja, Crítica (aprox), Alta
INTERVALO = 50       # Milisegundos entre frames

# --- 3. Ejecución y Visualización ---

# Diccionario para guardar los resultados de cada temperatura
resultados = {}

for T in TEMPERATURAS:
    print(f"Simulando para T = {T}...")
    resultados[T] = ejecutar_simulacion(N, T, PASOS_MC, FREC_GUARDADO)

# --- 4. Función para crear el GIF ---

def crear_animacion(frames, temp):
    fig, ax = plt.subplots()
    ax.axis("off")
    ax.set_title(f"Modelo de Ising (T = {temp})")

    im = ax.imshow(frames[0], cmap="binary", vmin=-1, vmax=+1)

    def update(i):
        im.set_data(frames[i])
        return [im]

    anim = FuncAnimation(fig, update, frames=len(frames),
                         interval=INTERVALO, blit=True)
    plt.close() # Para que no muestre una imagen estática extra
    return anim

# --- 5. Mostrar los resultados en el Notebook ---

# Por ejemplo, para ver la animación de la segunda temperatura de la lista:
temp_a_ver = TEMPERATURAS[1]
anim = crear_animacion(resultados[temp_a_ver], temp_a_ver)

# Esto renderiza el GIF directamente en Jupyter:
HTML(anim.to_jshtml())

Simulando para T = 1.5...
Simulando para T = 2.269...
Simulando para T = 4.0...
